# Preview and publish lightweight results

This notebook validates a Drive result bundle, previews the exact Git changes, and never stages or commits automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USERNAME = "Harryphan72007"
GITHUB_REPOSITORY = "aerial-object-detection-benchmark"
DEFAULT_BRANCH = "main"
DRIVE_ROOT = '/content/drive/MyDrive/visdrone_architecture_benchmark'
LOCAL_REPOSITORY = f'/content/{GITHUB_REPOSITORY}'
REPOSITORY_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git'
RESULT_BUNDLE_ID = '<SELECTED_RESULT_BUNDLE>'
GIT_USER_NAME = '<YOUR_NAME>'
GIT_USER_EMAIL = '<YOUR_EMAIL>'
assert GITHUB_USERNAME != '<MY_GITHUB_USERNAME>'


In [ ]:
import os, subprocess, sys
sys.path.insert(0, LOCAL_REPOSITORY)
from src.colab_setup import clone_or_update_repository, install_project, initialize_drive_directories, validate_drive_writable
clone_or_update_repository(REPOSITORY_URL, LOCAL_REPOSITORY, DEFAULT_BRANCH)
os.chdir(LOCAL_REPOSITORY)
install_project(LOCAL_REPOSITORY)
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(DRIVE_ROOT)


## Available Drive result bundles

In [ ]:
from pathlib import Path
bundles_root = Path(DRIVE_ROOT) / 'result_bundles'
available_bundles = sorted(p.name for p in bundles_root.iterdir() if p.is_dir())
available_bundles

## Dry-run preview

The command validates required files, run and checkpoint hashes, class/track compatibility, metrics, secrets, paths, extensions, and file sizes before copying anything.

In [ ]:
!python scripts/sync_results_to_repo.py --drive-root "$DRIVE_ROOT" --bundle-id "$RESULT_BUNDLE_ID" --repo-root "$LOCAL_REPOSITORY" --validate --dry-run --max-file-size-mb 20

## Copy approved files and inspect the diff

Run this cell only after the dry-run succeeds. It still does not stage or commit.

In [ ]:
!python scripts/sync_results_to_repo.py --drive-root "$DRIVE_ROOT" --bundle-id "$RESULT_BUNDLE_ID" --repo-root "$LOCAL_REPOSITORY" --validate --clean-target --max-file-size-mb 20
!git -C "$LOCAL_REPOSITORY" status --short
!git -C "$LOCAL_REPOSITORY" diff --stat
!git -C "$LOCAL_REPOSITORY" diff -- results/

## Identity and safe authentication

Set identity only for this repository. Preferred authentication is a temporary Colab Secret; never print, persist, or put the token in Git configuration, notebook output, Drive, or result manifests. Committing and pushing from a local computer is the safer alternative.

In [ ]:
from src.git_utils import configure_identity
configure_identity(LOCAL_REPOSITORY, GIT_USER_NAME, GIT_USER_EMAIL)
# If pushing from Colab, retrieve but never print the token:
# from google.colab import userdata
# github_token = userdata.get('GITHUB_TOKEN')


## Results branch, staging, validation, commit, and pull request

Inspect remote differences before choosing a rebase or merge. Do not force-push by default. Stage only approved paths with `git add results/ benchmark_data/`, never `git add .`. Verify `gh auth status` before an optional PR command.

Suggested commands:

```bash
git fetch origin
git checkout -B experiment-results origin/main
git status
git add results/ benchmark_data/
git diff --cached --stat
python scripts/validate_results.py --repo-results results/
git commit -m "results: add validated VisDrone benchmark evaluation"
git push -u origin experiment-results
gh auth status && gh pr create --base main --head experiment-results --title "Add latest VisDrone benchmark results" --body-file results/reports/pull_request_summary.md
```

In [ ]:
!python scripts/validate_results.py --repo-results results/